In [14]:
import yfinance as yf
from datetime import datetime,timedelta
ranges = ["2023-01-01","2025-11-30"]
symbol = "NVDA"
stock = yf.Ticker(symbol)
info = stock.info
sector = info.get("sectorKey", info.get("quoteType", "Uncategorized"))
history = stock.history(start=datetime.strptime(ranges[0], "%Y-%m-%d")-timedelta(days=365), end=datetime.strptime(ranges[1], "%Y-%m-%d"), interval="1d") # 2018 to give prophet data to base off of
window = history[ranges[0]:ranges[1]]["Close"] #training window
window

Date
2023-01-03 00:00:00-05:00     14.300682
2023-01-04 00:00:00-05:00     14.734249
2023-01-05 00:00:00-05:00     14.250735
2023-01-06 00:00:00-05:00     14.844141
2023-01-09 00:00:00-05:00     15.612370
                                ...    
2025-11-21 00:00:00-05:00    178.870041
2025-11-24 00:00:00-05:00    182.539841
2025-11-25 00:00:00-05:00    177.810104
2025-11-26 00:00:00-05:00    180.249954
2025-11-28 00:00:00-05:00    176.990143
Name: Close, Length: 730, dtype: float64

In [ ]:
def _make_cache_key(self, start_date, last_date, freq, data_series):
    """
    Create a stable key for the training window.
    Use start/end iso strings + freq + a short hash of the close values.
    """
    m = hashlib.sha1()
    # include length and last value to reduce collisions for small windows
    m.update(str(len(data_series)).encode())
    m.update(str(float(data_series.iloc[-1])).encode())
    # hash a slice of the series bytes (safe and fast)
    try:
        m.update(data_series.values.tobytes())
    except Exception:
        # fallback: hash string repr
        m.update(data_series.to_string().encode())
    return (start_date.isoformat(), last_date.isoformat(), freq, m.hexdigest())

def _cache_get(self, key):
    with self._cache_lock:
        item = self._prophet_cache.get(key)
        if not item:
            return None
        ts, value = item
        # TTL check
        if self._cache_ttl and (time.time() - ts) > self._cache_ttl:
            # expired
            del self._prophet_cache[key]
            return None
        # move to end (most recently used)
        self._prophet_cache.move_to_end(key)
        return value

def _cache_set(self, key, value):
    with self._cache_lock:
        self._prophet_cache[key] = (time.time(), value)
        self._prophet_cache.move_to_end(key)
        # evict oldest if over capacity
        while len(self._prophet_cache) > self._cache_max_items:
            self._prophet_cache.popitem(last=False)

def _prophetInit(self, history, lastDate, curPrice, histories, forward=90):
    prophetTrend = None
    prophetSigma = 0
    prophetSum = []

    # ensure histories is a dict (do this once before calling in outer code if possible)
    histories = ast.literal_eval(histories.replace('"', "'")) if isinstance(histories, str) else histories

    for h, nested in histories.items():
        start_date = lastDate - timedelta(days=h)
        # slice using index positions is faster; here we keep the original approach but use a series for hashing
        window = history[history.index > start_date]
        if len(window) < 20:
            continue

        # build cache key from start_date, lastDate, freq and the Close series
        close_series = window["Close"].copy()
        key = self._make_cache_key(start_date, lastDate, nested[1], close_series)

        # try cache
        cached_trend = self._cache_get(key)
        if cached_trend is not None:
            trend = cached_trend
        else:
            # prepare data for Prophet
            data = window.reset_index()[["Date", "Close"]].rename(columns={"Date": "ds", "Close": "y"})
            data["ds"] = data["ds"].dt.tz_localize(None)

            m = ph(daily_seasonality=False, yearly_seasonality=True, weekly_seasonality=True, n_changepoints=10)
            m.fit(data)

            future = m.make_future_dataframe(periods=forward, freq=nested[1])
            fcst = m.predict(future)
            trend = fcst.tail(forward + 1)["yhat"].values

            # store raw trend in cache (before offset)
            self._cache_set(key, trend)

        # Offset to align with current price (do not cache offset)
        prophetSum.append((trend + curPrice - trend[0]) * nested[0])

    if prophetSum:
        prophetTrend = np.sum(prophetSum, axis=0)

    return prophetTrend, prophetSigma

In [55]:
import yfinance as yf
import re

r = yf.Lookup("SPX").get_all(10)
r

,exchange,quoteType,rank,regularMarketChange,regularMarketPercentChange,regularMarketPrice,shortName,industryLink,industryName
symbol,,,,,,,,,
^SPX,WCB,index,20122,15.850098,0.231541,6861.350098,S&P 500 INDEX,NaN,NaN
SPXL,PCX,etf,20051,1.369995,0.621003,221.979996,Direxion Daily S&P 500 Bull 3X,NaN,NaN
SPXS,PCX,etf,20044,-0.223202,-0.629301,35.246799,Direxion Daily S&P 500 Bear 3X,NaN,NaN
SPXC,NYQ,equity,20013,2.975006,1.487057,203.035004,"SPX Technologies, Inc.",https://finance.yahoo.com/sector/industrials,Industrials
SPXU,PCX,etf,20007,-0.305000,-0.613374,49.415001,ProShares UltraPro Short S&P500,NaN,NaN
SPXSY,PNK,equity,20006,-0.480000,-1.035598,45.869999,SPIRAX GROUP PLC,https://finance.yahoo.com/sector/industrials,Industrials
SPXCY,OQX,equity,20006,-0.160000,-0.604229,26.320000,Singapore Exchange Ltd.,https://finance.yahoo.com/sector/financial_ser...,Financial Services
SPX.L,LSE,equity,20006,20.000000,0.293255,6840.000000,SPIRAX GROUP PLC ORD 26 12/13P,https://finance.yahoo.com/sector/industrials,Industrials
SPW0.F,FRA,equity,20005,-3.000000,-1.734104,170.000000,SPX Technologies Inc. R,https://finance.yahoo.com/sector/industrials,Industrials
